In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              silhouette_score, r2_score, mean_squared_error)
from xgboost import XGBClassifier, XGBRegressor

import warnings
warnings.filterwarnings("ignore")

sns.set(style="whitegrid")

In [ ]:
df = pd.read_csv(r"C:\Users\USER\Downloads\Country-data.csv")
df.head()

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.drop("country", axis=1).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
features = df.columns[1:]
df[features].hist(figsize=(14,10), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop("country", axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

In [ ]:
inertia = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K_range, inertia, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal k")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

sil_score = silhouette_score(X_scaled, df['KMeans_Cluster'])
print("Silhouette Score (KMeans):", sil_score)

df['KMeans_Cluster'].value_counts()

In [ ]:
cluster_summary = df.groupby('KMeans_Cluster')[['gdpp','income','child_mort','life_expec']].mean()
print(cluster_summary)

# Map clusters to meaningful labels based on gdpp (low to high)
order = cluster_summary['gdpp'].sort_values().index
label_map = {order[0]: 'Underdeveloped', order[1]: 'Developing', order[2]: 'Developed'}
df['Development_Status'] = df['KMeans_Cluster'].map(label_map)
df[['country','Development_Status']].head()

In [ ]:
pca = PCA(n_components=2)
pca_components = pca.fit_transform(X_scaled)
df['PCA1'], df['PCA2'] = pca_components[:,0], pca_components[:,1]

plt.figure(figsize=(8,6))
sns.scatterplot(x='PCA1', y='PCA2', hue='Development_Status', data=df, palette='Set2', s=80)
plt.title("K-Means Clusters (PCA Projection)")
plt.show()

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

print(df['DBSCAN_Cluster'].value_counts())

plt.figure(figsize=(8,6))
sns.scatterplot(x='PCA1', y='PCA2', hue='DBSCAN_Cluster', data=df, palette='tab10', s=80)
plt.title("DBSCAN Clusters (PCA Projection)")
plt.show()

In [ ]:
X_clf = X_scaled.copy()
y_clf = df['Development_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
print(X_train.shape, X_test.shape)

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)
rf_clf.fit(X_train, y_train)

rf_pred = rf_clf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

xgb_clf = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                         use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_clf.fit(X_train, y_train_enc)

xgb_pred = xgb_clf.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(y_test_enc, xgb_pred))
print(classification_report(y_test_enc, xgb_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=rf_clf.classes_, yticklabels=rf_clf.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
importances = pd.Series(rf_clf.feature_importances_, index=X_clf.columns).sort_values(ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Feature Importance (Random Forest)")
plt.show()

In [ ]:
X_reg = X_scaled.drop(columns=['gdpp'])
y_reg = df['gdpp']

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=300, random_state=42)
rf_reg.fit(Xr_train, yr_train)

rf_reg_pred = rf_reg.predict(Xr_test)
print("RF Regressor R2 Score:", r2_score(yr_test, rf_reg_pred))
print("RF Regressor RMSE:", np.sqrt(mean_squared_error(yr_test, rf_reg_pred)))

In [ ]:
xgb_reg = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)
xgb_reg.fit(Xr_train, yr_train)

xgb_reg_pred = xgb_reg.predict(Xr_test)
print("XGBoost Regressor R2 Score:", r2_score(yr_test, xgb_reg_pred))
print("XGBoost Regressor RMSE:", np.sqrt(mean_squared_error(yr_test, xgb_reg_pred)))

In [ ]:
param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1]
}

grid_xgb = GridSearchCV(XGBRegressor(random_state=42), param_grid,
                         cv=5, scoring='r2', n_jobs=-1)
grid_xgb.fit(Xr_train, yr_train)

print("Best Params:", grid_xgb.best_params_)
best_pred = grid_xgb.best_estimator_.predict(Xr_test)
print("Tuned XGBoost R2 Score:", r2_score(yr_test, best_pred))

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(yr_test, best_pred, alpha=0.7, color='teal')
plt.plot([yr_test.min(), yr_test.max()], [yr_test.min(), yr_test.max()], 'r--')
plt.xlabel("Actual GDPP")
plt.ylabel("Predicted GDPP")
plt.title("Actual vs Predicted GDPP (Tuned XGBoost)")
plt.show()

In [ ]:
print("===== CUSTOMER INTELLIGENCE SYSTEM SUMMARY =====")
print(f"Clustering (KMeans) Silhouette Score: {sil_score:.3f}")
print(f"Random Forest Classification Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(f"XGBoost Classification Accuracy: {accuracy_score(y_test_enc, xgb_pred):.3f}")
print(f"Random Forest Regression R2: {r2_score(yr_test, rf_reg_pred):.3f}")
print(f"XGBoost Regression R2 (tuned): {r2_score(yr_test, best_pred):.3f}")